<a href="https://colab.research.google.com/github/dozhash/computer_vision/blob/main/menu_detector_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
print("Menu Detector!")

Menu Detector!


In [2]:
# -------------------------
# Import Libraries
# -------------------------
from google.colab import drive
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.models import mobilenet_v2

from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import os
import numpy as np

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# Define Dataset Path
DATASET_PATH = '/content/drive/MyDrive/food101_dataset'

In [5]:
# Define Dataset Path

DATASET_PATH = '/content/drive/MyDrive/food101_dataset'
print('Dataset_path:', DATASET_PATH)


CUSTOM_CLASS_MAPPING = {
    'hamburger': 'hamburger',
    'hot_dog': 'hot_dog',
    'chocolate_cake': 'dessert', # label grouping | class consolidation
    'cheesecake': 'dessert', # label grouping | class consolidation
    'kebab': 'kebab_images',
    'pilaf': 'pilaf_images'
}



CLASSES = ['hamburger', 'hot_dog', 'dessert', 'kebab_images', 'pilaf_images']
CLASS_TO_IDX = {cls: i for i, cls in enumerate(CLASSES)}
NUM_CLASSES = len(CLASSES)

print('Classes:', CLASSES)
print('Class to Index:', CLASS_TO_IDX)
print('Number of Classes:', NUM_CLASSES)


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(), # H, W, C ==> C, H, W --> Channel: RGB
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

Dataset_path: /content/drive/MyDrive/food101_dataset
Classes: ['hamburger', 'hot_dog', 'dessert', 'kebab_images', 'pilaf_images']
Class to Index: {'hamburger': 0, 'hot_dog': 1, 'dessert': 2, 'kebab_images': 3, 'pilaf_images': 4}
Number of Classes: 5


In [6]:
# -------------------------
# Custom Dataset Class
# -------------------------

class FoodDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        # print('images_length', len(self.images))
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        # print('image_path', img_path)

        label = self.labels[idx]
        # print('label', label)

        try:
            image = Image.open(img_path)

            if image.mode == "P" or image.mode == "RGBA": # PNG | GIF | RGBA
                image = image.convert("RGBA").convert("RGB")
            else:
                image = image.convert("RGB")

        except (UnidentifiedImageError, OSError):
            print(f"Skipping broken image: {img_path}")
            return self.__getitem__((idx + 1) % len(self.images))

        if self.transform:
            image = self.transform(image)

        return image, label

In [7]:
# ==========================
# Gather and Split Data
# ==========================

all_images = []

for original_class, mapped_class in CUSTOM_CLASS_MAPPING.items():
    class_path = os.path.join(DATASET_PATH, original_class)
    print('class_path:', class_path)

    if not os.path.exists(class_path):
        print(f"Warning: {class_path} not found")
        continue

    for img in os.listdir(class_path):
        if img.endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(class_path, img)
            all_images.append((full_path, CLASS_TO_IDX[mapped_class]))

np.random.shuffle(all_images)

split = int(0.8 * len(all_images))
train_data = all_images[:split]
val_data = all_images[split:]

train_images, train_labels = zip(*train_data)
val_images, val_labels = zip(*val_data)

# print('all_images:', all_images)

dataset = FoodDataset(train_images, train_labels)
print(len(dataset))

img, lbl = dataset[0]

class_path: /content/drive/MyDrive/food101_dataset/hamburger
class_path: /content/drive/MyDrive/food101_dataset/hot_dog
class_path: /content/drive/MyDrive/food101_dataset/chocolate_cake
class_path: /content/drive/MyDrive/food101_dataset/cheesecake
class_path: /content/drive/MyDrive/food101_dataset/kebab
class_path: /content/drive/MyDrive/food101_dataset/pilaf
3356


In [8]:
train_dataset = FoodDataset(train_images, train_labels, transform=transform)
val_dataset = FoodDataset(val_images, val_labels, transform=transform)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)  # thread | parallel loading for speed

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [10]:
# pretrained model

model = mobilenet_v2(weights="IMAGENET1K_V1")  # pretrained model | lightweight | CNN | 1000 class | million

model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)  # fine-tuning | backbone | model layer freeze

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 85.1MB/s]


In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

model = model.to(device)

device cuda


In [12]:
criterion = nn.CrossEntropyLoss()  # Loss Function | '70%' burger, '30%' pilaf

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)  # weight

torch.backends.cudnn.benchmark = True  # Benchmark Setting | Trick | 10%-20%

In [13]:
# -------------------------
# Training Loop
# -------------------------

NUM_EPOCHS = 10
best_accuracy = 0.0

for epoch in range(NUM_EPOCHS):
    model.train()  # train mode
    running_loss = 0.0

    for images, labels in train_loader:  # Forward and Backward (Backpropagation)
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()  # zero the gradient

        outputs = model(images)  # Forward Pass

        loss = criterion(outputs, labels)  # Calculate Loss

        loss.backward()

        optimizer.step()  # Adam optimizer

        running_loss += loss.item()  # Track Loss

    # Validation
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total # Calculate Validation Accuracy

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Loss: {running_loss/len(train_loader):.4f}, "
        f"Val Accuracy: {val_acc:.2f}%"
    )

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        torch.save(model.state_dict(), "/content/menu_detector.pth")
        print("Saved new best model!")

Epoch [1/10] Loss: 0.4846, Val Accuracy: 85.10%
Saved new best model!
Epoch [2/10] Loss: 0.3101, Val Accuracy: 88.20%
Saved new best model!
Epoch [3/10] Loss: 0.2594, Val Accuracy: 89.63%
Saved new best model!
Epoch [4/10] Loss: 0.1938, Val Accuracy: 88.08%
Epoch [5/10] Loss: 0.1578, Val Accuracy: 85.34%
Epoch [6/10] Loss: 0.1389, Val Accuracy: 89.03%
Epoch [7/10] Loss: 0.1467, Val Accuracy: 89.15%
Epoch [8/10] Loss: 0.1194, Val Accuracy: 90.82%
Saved new best model!
Epoch [9/10] Loss: 0.1076, Val Accuracy: 87.84%
Epoch [10/10] Loss: 0.1521, Val Accuracy: 86.41%
